In [ ]:
!pip install apache-airflow

INFO: pip is looking at multiple versions of cadwyn to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.5/607.5 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.4/97.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.0/163.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 10.3 MB/s 

In [ ]:
!airflow version

3.3.2


In [ ]:
import os

os.environ["AIRFLOW_HOME"] = "/content/airflow"

!mkdir -p /content/airflow/dags

print("Airflow folder created successfully.")

Airflow folder created successfully.


In [ ]:
!airflow db migrate

2026-09-17T14:58:52.898678Z [info     ] Performing upgrade to the metadata database [airflow.cli.commands.db_command] loc=db_command.py:134 url=sqlite:////content/airflow/airflow.db
2026-09-17T14:58:53.270239Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-09-17T14:58:53.271130Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-09-17T14:58:54.561576Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-09-17T14:58:54.561816Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-09-17T14:58:54.562330Z [info     ] Creating Airflow database tables from the ORM [airflow.utils.db] loc=db.py:749
2026-09-17T14:58:54.562541Z [info     ] Creating global lock context   [airflow.utils.db] loc=db.py:730
2026-09-17T14:58:54.562725Z [info     ] Pool status: Pool size: 5  Connections in pool: 0 Current Overflow:

In [ ]:
%%writefile /content/data_extraction.py

import pandas as pd
import requests

# Extract data from API
def extract_api_data(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json()
    return pd.json_normalize(data)


# Extract data from CSV
def extract_csv_data(file_path):
    return pd.read_csv(file_path)


# API source
api_url = "https://jsonplaceholder.typicode.com/users"

api_df = extract_api_data(api_url)

# Select required API columns
api_df = api_df[
    ["id", "name", "email", "company.name"]
]

api_df.rename(
    columns={"company.name": "company"},
    inplace=True
)


# Create sample CSV file
csv_file = "/content/locations.csv"

pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "city": ["Mumbai", "Pune", "Delhi", "Chennai", "Bangalore"],
    "country": ["India", "India", "India", "India", "India"]
}).to_csv(csv_file, index=False)


# Read CSV
csv_df = extract_csv_data(csv_file)


# Merge API and CSV data
merged_df = pd.merge(
    api_df,
    csv_df,
    on="id",
    how="inner"
)


# Save final data
merged_df.to_csv(
    "/content/cleaned_warehouse_profiles.csv",
    index=False
)

print("ETL extraction completed successfully!")
print(merged_df)

Writing /content/data_extraction.py


In [ ]:
%%writefile /content/airflow/dags/university_etl_orchestration.py

from datetime import datetime, timedelta

from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.empty import EmptyOperator

default_args = {
    "owner": "data_engineering_lab",
    "depends_on_past": False,
    "start_date": datetime(2026, 1, 1),
    "retries": 1,
    "retry_delay": timedelta(minutes=5),
}

with DAG(
    dag_id="university_etl_orchestration",
    default_args=default_args,
    description="ETL pipeline orchestration",
    schedule=timedelta(days=1),
    catchup=False,
) as dag:

    start_pipeline = EmptyOperator(
        task_id="start_pipeline"
    )

    execute_extraction = BashOperator(
        task_id="run_extraction_script",
        bash_command="python /content/data_extraction.py",
    )

    pipeline_complete = BashOperator(
        task_id="log_pipeline_success",
        bash_command='echo "ETL Execution completed successfully"',
    )

    start_pipeline >> execute_extraction >> pipeline_complete

Writing /content/airflow/dags/university_etl_orchestration.py


In [ ]:
!airflow dags list

No data found


In [ ]:
!ls -l /content/airflow/dags/

total 4
-rw-r--r-- 1 root root 981 Sep 17 15:01 university_etl_orchestration.py


In [ ]:
!airflow config get-value core dags_folder


/content/airflow/dags


In [ ]:
!airflow dags list-import-errors

No data found


In [ ]:
import subprocess

log = open("/content/dag_processor.log", "w")

process = subprocess.Popen(
    ["airflow", "dag-processor"],
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("DAG processor started. Process ID:", process.pid)

DAG processor started. Process ID: 4891


In [ ]:
!airflow dags list

             |             |             |           |             | bundle_vers
dag_id       | fileloc     | owners      | is_paused | bundle_name | ion        
=============+=============+=============+===========+=============+============
aggregate_re | /usr/local/ | airflow     | True      | example_dag | None       
gional_sales | lib/python3 |             |           | s           |            
             | .13/dist-pa |             |           |             |            
             | ckages/airf |             |           |             |            
             | low/example |             |           |             |            
             | _dags/examp |             |           |             |            
             | le_asset_pa |             |           |             |            
             | rtition.py  |             |           |             |            
asset1_produ | /usr/local/ | airflow     | True      | example_dag | None       
cer          | lib/python3 |

In [ ]:
!airflow dags list | grep university_etl_orchestration

university_etl_orchestration                     | /content/airflow/dags/university_etl_orchestrat | data_engineering_lab | True      | dags-folder                                     | None          


In [ ]:
!airflow dags show university_etl_orchestration

digraph university_etl_orchestration {
	graph [label=university_etl_orchestration labelloc=t rankdir=LR]
	log_pipeline_success [color="#000000" fillcolor="#f0ede4" label=log_pipeline_success shape=rectangle style="filled,rounded"]
	run_extraction_script [color="#000000" fillcolor="#f0ede4" label=run_extraction_script shape=rectangle style="filled,rounded"]
	start_pipeline [color="#000000" fillcolor="#e8f7e4" label=start_pipeline shape=rectangle style="filled,rounded"]
	run_extraction_script -> log_pipeline_success
	start_pipeline -> run_extraction_script
}



In [ ]:
!airflow dags trigger university_etl_orchestration

2026-09-17T15:09:36.073862Z [info     ] creating dag run               [airflow.serialization.definitions.dag] loc=dag.py:615 logical_date=None partition_key=None run_after=datetime.datetime(2026, 9, 17, 15, 9, 36, 60298, tzinfo=Timezone('UTC')) run_id=manual__2026-09-17T15:09:36.060298+00:00
     |      |      |      |      |      | last |      |      |     |      |     
     |      |      | data | data |      | _sch |      |      |     |      | trig
     |      |      | _int | _int |      | edul |      |      | sta |      | geri
     |      | dag_ | erva | erva |      | ing_ | logi |      | rt_ |      | ng_u
     | dag_ | run_ | l_st | l_en | end_ | deci | cal_ | run_ | dat | stat | ser_
conf | id   | id   | art  | d    | date | sion | date | type | e   | e    | name
=====+======+======+======+======+======+======+======+======+=====+======+=====
{}   | univ | manu | None | None | None | None | None | manu | Non | queu | root
     | ersi | al__ |      |      |      |      |      | al

In [ ]:
!airflow dags list-runs -d university_etl_orchestration

Usage: airflow [-h] GROUP_OR_COMMAND ...

Positional Arguments:
  GROUP_OR_COMMAND

    Groups
      assets            Manage assets
      backfill          Manage backfills
      config            View configuration
      connections       Manage connections
      dags              Manage DAGs
      db                Database operations
      db-manager        Manage externally connected database managers
      jobs              Manage jobs
      partitions        Manage Dag run partition metadata
      pools             Manage pools
      providers         Display providers
      state-store       Manage task and asset state storage
      tasks             Manage tasks
      teams             Manage teams
      variables         Manage variables

    Commands:
      api-server        Start an Airflow API server instance
      cheat-sheet       Display cheat sheet
      dag-processor     Start a dag processor instance
      info              Show information about current Airflow and 

In [ ]:
!airflow dags state university_etl_orchestration 2026-09-17T15:09:36.073862+00:00


None


In [ ]:
!airflow tasks --help

Usage: airflow tasks [-h] COMMAND ...

Manage tasks

Positional Arguments:
  COMMAND
    clear               Clear a set of task instance, as if they never ran
    failed-deps         Returns the unmet dependencies for a task instance
    list                List the tasks within a DAG
    render              Render a task instance's template(s)
    state               Get the status of a task instance
    states-for-dag-run  Get the status of all task instances in a dag run
    test                Test a task instance

Options:
  -h, --help            show this help message and exit


In [ ]:
!airflow tasks states-for-dag-run --help

Usage: airflow tasks states-for-dag-run [-h] [-o (table, json, yaml, plain)]
                                        [-v]
                                        dag_id logical_date_or_run_id

Get the status of all task instances in a dag run

Positional Arguments:
  dag_id                The id of the dag
  logical_date_or_run_id
                        The logical date of the DAG or run_id of the DAGRun

Options:
  -h, --help            show this help message and exit
  -o, --output (table, json, yaml, plain)
                        Output format. Allowed values: json, yaml, plain, table (default: table)
  -v, --verbose         Make logging output more verbose
